_This notebook pixelizes the habitat map to the Sentinel-2 raster. It also extends the habitat index to keep track of each individual habitat pixel._

_At the time of writing this notebook the following was known:_ <br>
- Gelderland hab kart is True
- Website plusHR is True
- the _plusOW gdfs have problems

Taking into account the total pixel counts, the new tmp2 selection, it is decided to pixelize the following three gpkgs:
- Gelderland Hab kart -> FC +80% tmp2
- Website Hab kart -> FC +80% tmp2
- OW all years selected

## Importing functions

In [1]:
from functions.gpkg_funcs import (
    import_gpkg_func,
    pixelize_gpkg_func,
)

from functions.idx_to_gdf_or_plot_funcs import (
    idx_df_to_gdf_func,
)

## Setting directory

In [2]:
from paths.OG_paths import (
    #######################################
    #            00_common_dfs            #
    #######################################
    # |    n01_habitat_reference_dfs      |
    # +===================================+ 
    habitat_reference__WD__df_path,
    habitat_reference__s__df_path,


    #######################################
    #     01_pre_processing_hab_kart      #
    #######################################
    # |  n01_T0_habitat_gpkg_processing   |
    # +===================================+ 
    # --- Habitat kart Processed GPKG --- #
    habitat_kart__processed__gelderland__RD__gpkg_path,
    habitat_kart__processed__website_plusHR__RD__gpkg_path,


    #######################################
    #      01_pre_processing_sat_obs      #
    #######################################
    # |        n01_exp_sat_raster         |
    # +===================================+
    s2_basisraster__10m__RD__gpkg_path,


    #######################################
    #  02_hab_kart_selection_and_division #
    #######################################
     # |     n01_first_data_division      |
    # +===================================+
    idx__FC_p80__division__gelderland__tmp2__df_path,
    idx__FC_p80__division__website_plusHR__tmp2__df_path,

    # +===================================+
    # |    n03_add_LGN_OW_to_hab_kart     |
    # +===================================+
    # --- lgn OW stacked GPKG --- #
    lgn_plusOW__n2000__stacked__gelderland__RD__gpkg_path,

    # +===================================+
    # |       n05_pixelize_hab_kart       |
    # +===================================+
    # --- FC p80 tmp2 GPKGs --- #
    habitat_kart__FC_p80_tmp2_polys__gelderland__RD__gpkg_path,
    habitat_kart__FC_p80_tmp2_polys__website_plusHR__RD__gpkg_path,

    # --- Habitat kart pixeled for inspection GPKGs --- #
    # pixeled GPKGs
    habitat_kart__FC_p80_tmp2_inspection__gelderland__RD__gpkg_path,
    habitat_kart__FC_p80_tmp2_inspection__website_plusHR__RD__gpkg_path,
    OW_all_years__inspection__lgn__RD__gpkg_path,
)

## Importing packages

In [3]:
import pandas as pd

## Import files

In [4]:
# --- Selection reference dfs ---
habitat_reference__WD__df = pd.read_pickle(habitat_reference__WD__df_path)
habitat_reference__s__df = pd.read_pickle(habitat_reference__s__df_path)

In [5]:
# --- Habitat kart Processed GPKGs ---
# Gelderland 
habitat_kart__processed__gelderland__RD__gdf = import_gpkg_func(
    gdf_path=habitat_kart__processed__gelderland__RD__gpkg_path, 
    layer='habitat_kart__processed__gelderland__RD__layer', 
    index_col='index'
)

# Website plus hr 
habitat_kart__processed__website_plusHR__RD__gdf = import_gpkg_func(
    gdf_path=habitat_kart__processed__website_plusHR__RD__gpkg_path,
    layer='habitat_kart__processed__website_plusHR__RD__layer',
    index_col='index'
)

# All years OW
lgn_plusOW__n2000__stacked__gelderland__RD__gdf = import_gpkg_func(
    gdf_path=lgn_plusOW__n2000__stacked__gelderland__RD__gpkg_path, 
    layer='lgn_plusOW__n2000__stacked__gelderland__RD__layer', 
    index_col='index'
)

c:\Users\NL1G3K\Desktop\Vegetation_quality_monitoring\.pixi\envs\default\Lib\site-packages\pyogrio\core.py:34: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


In [6]:
# --- IDX FC p80 tmp2 --- #
idx__FC_p80__division__gelderland__tmp2__df = pd.read_pickle(idx__FC_p80__division__gelderland__tmp2__df_path)
idx__FC_p80__division__website_plusHR__tmp2__df = pd.read_pickle(idx__FC_p80__division__website_plusHR__tmp2__df_path)

## Build a FC p80 tmp2 Hab kart

In [7]:
# FC Gelderland
habitat_kart__FC_p80_tmp2_polys__gelderland__RD__gdf = idx_df_to_gdf_func(
    idx_df=idx__FC_p80__division__gelderland__tmp2__df,
    source_gdf=habitat_kart__processed__gelderland__RD__gdf,
    out_gpkg=habitat_kart__FC_p80_tmp2_polys__gelderland__RD__gpkg_path,
    gdf_cols_to_keep=habitat_kart__processed__gelderland__RD__gdf.columns,
    single_layer_name="habitat_kart__FC_p80_tmp2_polys__gelderland__RD__layer",
)

In [8]:
# FC Website_plusHR
habitat_kart__FC_p80_tmp2_polys__website_plusHR__RD__gdf = idx_df_to_gdf_func(
    idx_df=idx__FC_p80__division__website_plusHR__tmp2__df,
    source_gdf=habitat_kart__processed__website_plusHR__RD__gdf,
    out_gpkg=habitat_kart__FC_p80_tmp2_polys__website_plusHR__RD__gpkg_path,
    gdf_cols_to_keep=habitat_kart__processed__gelderland__RD__gdf.columns,
    single_layer_name="habitat_kart__FC_p80_tmp2_polys__website_plusHR__RD__layer",
)

## Clip the processed GDF to the Sentinel raster and save

In [12]:
# Hab kart Gelderland clip to raster (only full pixels) -> very time consuming!
habitat_kart__FC_p80_tmp2_inspection__gelderland__RD__gdf = pixelize_gpkg_func(
    pixels_gpkg=s2_basisraster__10m__RD__gpkg_path,
    clip_gpkg=habitat_kart__FC_p80_tmp2_polys__gelderland__RD__gpkg_path,
    out_gpkg=habitat_kart__FC_p80_tmp2_inspection__gelderland__RD__gpkg_path,
    out_layer="habitat_kart__FC_p80_tmp2_inspection__gelderland__RD__layer",
    full_pixels_only=True,
)

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.


In [10]:
# Hab kart Website clip to raster (only full pixels) -> very time consuming!
habitat_kart__FC_p80_tmp2_inspection__website_plusHR__RD__gdf = pixelize_gpkg_func(
    pixels_gpkg=s2_basisraster__10m__RD__gpkg_path,
    clip_gpkg=habitat_kart__FC_p80_tmp2_polys__website_plusHR__RD__gpkg_path,
    out_gpkg=habitat_kart__FC_p80_tmp2_inspection__website_plusHR__RD__gpkg_path,
    out_layer="habitat_kart__FC_p80_tmp2_inspection__website_plusHR__RD__layer",
    full_pixels_only=True,
)

c:\Users\NL1G3K\Desktop\Vegetation_quality_monitoring\.pixi\envs\default\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Cannot find tms_NZTM2000.json (GDAL_DATA is not defined)
  ogr_write(


In [11]:
# OW all years clip to raster (full) -> very time consuming!
OW_all_years__inspection__lgn__RD__gdf = pixelize_gpkg_func(
    pixels_gpkg=s2_basisraster__10m__RD__gpkg_path,
    clip_gpkg=lgn_plusOW__n2000__stacked__gelderland__RD__gpkg_path,
    out_gpkg=OW_all_years__inspection__lgn__RD__gpkg_path,
    out_layer="OW_all_years__inspection__lgn__RD__layer",
    full_pixels_only=True,
)

End of the Notebook